In [2]:
# Cell 1: Imports and Setup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager
import time

def init_driver(headless=True):
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--disable-gpu")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--log-level=3")
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)


In [ ]:
# Cell 2: Core Scraper
def scrape_meta_quest_app_links(
    url="https://www.meta.com/en-gb/experiences/section/3955297897903802/", # https://www.meta.com/en-gb/experiences/section/3878844519028756/ is for browse all
    max_no_new_rounds=8,
    scroll_pause=2.5,
    target_min_count=900,
    max_scroll_rounds=100,
    headless=True
):
    driver = init_driver(headless=headless)
    driver.get(url)
    time.sleep(5)

    # Handle language modal
    try:
        print("[INFO] Checking for language selection modal...")
        lang_button = driver.find_element(By.XPATH, '//button[contains(., "English")]')
        lang_button.click()
        time.sleep(3)
        print("[INFO] Language selected.")
    except:
        print("[INFO] No language modal found or already dismissed.")

    collected_urls = set()
    all_seen_urls = []
    last_seen_count = 0
    no_new_rounds = 0
    scroll_round = 0

    print("[INFO] Starting scroll scraping...")
    while scroll_round < max_scroll_rounds:
        scroll_round += 1
        driver.execute_script("window.scrollBy(0, window.innerHeight * 0.8);")
        time.sleep(scroll_pause)

        # Get all matching app URLs (excluding section/category)
        anchors = driver.find_elements(By.XPATH, '//a[starts-with(@href, "/experiences/")]')
        new_urls = []
        for a in anchors:
            href = a.get_attribute("href")
            if not href:
                continue
            if any(skip in href for skip in ["section", "category"]):
                continue
            full_url = "https://www.meta.com" + href if href.startswith("/") else href
            if full_url not in collected_urls:
                collected_urls.add(full_url)
                new_urls.append(full_url)

        # Logging
        if new_urls:
            print(f"[ROUND {scroll_round}] {len(new_urls)} new URLs found. Total: {len(collected_urls)}")
            no_new_rounds = 0
        else:
            no_new_rounds += 1
            print(f"[ROUND {scroll_round}] No new URLs. ({no_new_rounds}/{max_no_new_rounds})")

        if no_new_rounds >= max_no_new_rounds:
            print("[INFO] No new URLs for several rounds. Stopping.")
            break

    driver.quit()

    print(f"[DONE] Final app count: {len(collected_urls)} (expected ≥ {target_min_count})")
    if len(collected_urls) < target_min_count:
        print("[WARN] Fewer apps than expected — you may want to adjust scrolling or delays.")
    return sorted(list(collected_urls))


In [4]:
# Cell 3: Run and Save
app_links = scrape_meta_quest_app_links(headless=False)

# Optionally save to file
with open("meta_quest_app_links.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(app_links))

print(f"[SAVED] {len(app_links)} links saved to meta_quest_app_links.txt")


[INFO] Checking for language selection modal...
[INFO] No language modal found or already dismissed.
[INFO] Starting scroll scraping...
[ROUND 1] 30 new URLs found. Total: 30
[ROUND 2] 16 new URLs found. Total: 46
[ROUND 3] No new URLs. (1/8)
[ROUND 4] 16 new URLs found. Total: 62
[ROUND 5] 16 new URLs found. Total: 78
[ROUND 6] No new URLs. (1/8)
[ROUND 7] 15 new URLs found. Total: 93
[ROUND 8] 16 new URLs found. Total: 109
[ROUND 9] No new URLs. (1/8)
[ROUND 10] 16 new URLs found. Total: 125
[ROUND 11] 15 new URLs found. Total: 140
[ROUND 12] No new URLs. (1/8)
[ROUND 13] 16 new URLs found. Total: 156
[ROUND 14] No new URLs. (1/8)
[ROUND 15] 32 new URLs found. Total: 188
[ROUND 16] 15 new URLs found. Total: 203
[ROUND 17] No new URLs. (1/8)
[ROUND 18] 16 new URLs found. Total: 219
[ROUND 19] 16 new URLs found. Total: 235
[ROUND 20] No new URLs. (1/8)
[ROUND 21] 16 new URLs found. Total: 251
[ROUND 22] 16 new URLs found. Total: 267
[ROUND 23] No new URLs. (1/8)
[ROUND 24] 16 new URLs 

In [5]:
# Optionally save to file
with open("meta_quest_app_links_full.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(app_links))